# tracewise — Kaggle experiments (M1: dataset + preprocessing)

Knowledge tracing on **ASSISTments 2009 skill-builder**. This notebook clones the GitHub repo, copies the attached Kaggle dataset into the expected path, and builds train/val/test student sequences.

BKT / DKT training cells are at the bottom and are **off** until you flip the flags. Do not start them before M1 sanity checks pass.

### Before Run All
1. Settings → Accelerator: **GPU** (T4 is fine; M1 is CPU-only, GPU is for later DKT)
2. Settings → Internet: **ON** (needed to `git clone`)
3. **Add Data** → search `ASSISTments 2009 skill builder` or `skill builder 2009-2010` → Add
4. Repo must be reachable: `https://github.com/HusseinHanafy207/tracewise` (public, or add a `GITHUB_TOKEN` secret)

Expected raw file after the copy step: `tracewise/data/raw/skill_builder_data.csv`


## 0. Run flags

In [ ]:
# M1 is the only thing that should run on first upload.
RUN_PREPROCESS = True
RUN_BKT = False   # M2 — leave False until preprocessing looks right
RUN_DKT = False   # M3 — leave False until BKT has numbers


## 1. Clone tracewise + install

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def run(cmd, cwd=None):
    printable = []
    for part in cmd:
        printable.append("***" if part.startswith("https://") and "@github.com" in part else part)
    print("+", " ".join(printable))
    subprocess.check_call(cmd, cwd=cwd)


root = Path("/kaggle/working")
repo = root / "tracewise"
clone_url = "https://github.com/HusseinHanafy207/tracewise.git"

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    clone_url = f"https://{token}@github.com/HusseinHanafy207/tracewise.git"
    print("Using GITHUB_TOKEN from Kaggle secrets")
except Exception:
    print("No GITHUB_TOKEN secret; cloning public HTTPS URL")

if repo.exists():
    run(["git", "-C", str(repo), "pull"])
else:
    run(["git", "clone", clone_url, str(repo)])

# Torch / sklearn / pandas / numpy are already on the Kaggle image.
# Do not reinstall torch from requirements.txt.
py = sys.executable
run([py, "-m", "pip", "install", "-q", "pyyaml", "tqdm"])

os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

print("cwd:", Path.cwd())
print("repo:", repo)
print("python:", sys.version.split()[0])

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


## 2. Attach ASSISTments and copy CSV

Kaggle mounts added datasets under `/kaggle/input/<dataset-slug>/`. This cell finds a skill-builder CSV and copies it to the path `src/data/preprocess.py` expects.

If discovery fails, set `CSV_OVERRIDE` to the full path printed by the `/kaggle/input` listing.


In [ ]:
from pathlib import Path
import shutil

import pandas as pd

CSV_OVERRIDE = None  # e.g. Path("/kaggle/input/your-slug/skill_builder_data.csv")

REQUIRED_COLS = {"user_id", "order_id", "skill_id", "correct"}
NAME_HINTS = ("skill_builder", "skillbuilder", "assistment", "assistments")


def list_input_files():
    root = Path("/kaggle/input")
    if not root.exists():
        print("No /kaggle/input — did you Add Data?")
        return []
    files = [p for p in root.rglob("*") if p.is_file()]
    print(f"Files under /kaggle/input ({len(files)}):")
    for p in files[:80]:
        print(" ", p)
    if len(files) > 80:
        print(f"  ... {len(files) - 80} more")
    return files


def score_csv(path: Path) -> int:
    name = path.name.lower()
    score = 0
    if path.suffix.lower() not in {".csv", ".txt"} and not name.endswith(".csv.gz"):
        return -1
    for hint in NAME_HINTS:
        if hint in name:
            score += 5
    if "skill_builder_data" in name:
        score += 10
    if "corrected" in name:
        score += 2
    return score


def peek_columns(path: Path):
    try:
        df = pd.read_csv(path, nrows=3, encoding="utf-8", low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(path, nrows=3, encoding="latin-1", low_memory=False)
    return [str(c) for c in df.columns]


files = list_input_files()
if CSV_OVERRIDE is not None:
    src = Path(CSV_OVERRIDE)
else:
    ranked = sorted(files, key=score_csv, reverse=True)
    ranked = [p for p in ranked if score_csv(p) > 0]
    if not ranked:
        csvs = [p for p in files if p.suffix.lower() == ".csv"]
        raise FileNotFoundError(
            "Could not find an ASSISTments skill-builder CSV under /kaggle/input. "
            "Add Data → search 'ASSISTments 2009 skill builder', then re-run. "
            f"CSV files seen: {csvs}"
        )
    src = ranked[0]
    print("Selected:", src, "(score", score_csv(src), ")")

cols = peek_columns(src)
print("columns:", cols)
missing = REQUIRED_COLS - set(cols)
if missing:
    raise ValueError(
        f"{src} is missing required columns {missing}. "
        "This is probably not the 2009 skill-builder file."
    )

dest = Path("/kaggle/working/tracewise/data/raw/skill_builder_data.csv")
dest.parent.mkdir(parents=True, exist_ok=True)
if src.resolve() != dest.resolve():
    shutil.copy2(src, dest)
print("raw CSV:", dest, "size_mb:", round(dest.stat().st_size / 1e6, 2))


## 3. Preprocess → student sequences + splits

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

processed = Path("/kaggle/working/tracewise/data/processed/meta.json")

if not RUN_PREPROCESS:
    print("RUN_PREPROCESS=False; skip")
elif processed.exists():
    print("Processed splits already exist; skip. Delete data/processed to rebuild.")
    print(processed.read_text())
else:
    subprocess.check_call(
        [sys.executable, "scripts/run_preprocessing.py", "--config", "configs/config.yaml"],
        cwd="/kaggle/working/tracewise",
    )


## 4. Sanity checks (M1 deliverable)

In [ ]:
from pathlib import Path
import json
import pickle

import matplotlib.pyplot as plt
import numpy as np

REPO = Path("/kaggle/working/tracewise")
processed = REPO / "data/processed"

with open(processed / "meta.json", encoding="utf-8") as f:
    meta = json.load(f)

print("meta.json")
print(json.dumps(meta, indent=2))

splits = {}
for name in ("train", "val", "test"):
    with open(processed / f"{name}.pkl", "rb") as f:
        splits[name] = pickle.load(f)

print()
print("split | students | interactions | mean_len | median_len | pct_correct")
all_lens = []
all_correct = []
for name, seqs in splits.items():
    lens = [len(s.skill_ids) for s in seqs]
    correct = [c for s in seqs for c in s.correct]
    all_lens.extend(lens)
    all_correct.extend(correct)
    print(
        f"{name:5} | {len(seqs):8d} | {sum(lens):12d} | "
        f"{np.mean(lens):8.1f} | {np.median(lens):10.1f} | {100 * np.mean(correct):10.2f}%"
    )

correct_arr = np.asarray(all_correct)
print()
print("overall students:", sum(len(v) for v in splits.values()))
print("overall pct_correct: {:.2f}%".format(100 * correct_arr.mean()))
print("min/max sequence length:", min(all_lens), max(all_lens))

# student-level split: no user_id should appear in two splits
ids = {name: {s.user_id for s in seqs} for name, seqs in splits.items()}
overlap_tv = ids["train"] & ids["val"]
overlap_tt = ids["train"] & ids["test"]
overlap_vt = ids["val"] & ids["test"]
print("user_id overlap train/val/test:", len(overlap_tv), len(overlap_tt), len(overlap_vt))
assert not (overlap_tv or overlap_tt or overlap_vt), "Student leakage across splits"

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(all_lens, bins=40, color="#3b6ea5")
axes[0].set_title("Sequence length (interactions / student)")
axes[0].set_xlabel("length")
axes[0].set_ylabel("students")
n_inc = int((correct_arr == 0).sum())
n_cor = int((correct_arr == 1).sum())
axes[1].bar(["incorrect", "correct"], [n_inc, n_cor], color=["#b35c5c", "#3b8a5a"])
axes[1].set_title("Response labels")
plt.tight_layout()
plt.show()

print()
print("M1 done. Next: set RUN_BKT=True (M2), then RUN_DKT=True (M3).")


## 5. Later: BKT / DKT (do not enable on first upload)

These call the existing repo scripts. Leave the flags in cell 0 as `False` until M1 looks right.

M2 BKT grid search can take a while on CPU. M3 DKT should use the GPU accelerator.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = "/kaggle/working/tracewise"
cfg = "configs/config.yaml"

if RUN_BKT:
    subprocess.check_call(
        [sys.executable, "scripts/fit_bkt.py", "--config", cfg],
        cwd=REPO,
    )
else:
    print("RUN_BKT=False; skip M2")

if RUN_DKT:
    subprocess.check_call(
        [sys.executable, "scripts/train_dkt.py", "--config", cfg],
        cwd=REPO,
    )
else:
    print("RUN_DKT=False; skip M3")

ckpt = Path(REPO) / "results/checkpoints"
print("checkpoints:", sorted(p.name for p in ckpt.glob("*")) if ckpt.exists() else "(none yet)")
